In [10]:
import pandas as pd
import json
import os

# Create output directory if it doesn't exist
output_dir = '01_data/processed'
os.makedirs(output_dir, exist_ok=True)


### 1. Load both files

We will load `ledger.csv` and `gateway.csv` into separate pandas DataFrames.

In [15]:
# Load the datasets
ledger_df = pd.read_csv('ledger.csv')
gateway_df = pd.read_csv('gateway.csv')

print("Ledger DataFrame Head:")
display(ledger_df.head())
print("\nGateway DataFrame Head:")
display(gateway_df.head())

Ledger DataFrame Head:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,850.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R004,2026-03-02,M003,2100.0,success,Card
4,R005,2026-03-03,M004,7200.0,success,Card



Gateway DataFrame Head:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,900.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R005,2026-03-03,M004,7200.0,failed,Card
4,R006,2026-03-03,M002,950.0,success,UPI


### 2. Check duplicates and nulls

It's important to understand the data quality before proceeding with reconciliation. We'll check for duplicate rows and missing values in both DataFrames.

In [16]:
# Check for duplicates in ledger_df
duplicate_ledger = ledger_df.duplicated().sum()
print(f"Number of duplicate rows in ledger_df: {duplicate_ledger}")

# Check for nulls in ledger_df
print("\nNull values in ledger_df:")
display(ledger_df.isnull().sum())

# Check for duplicates in gateway_df
duplicate_gateway = gateway_df.duplicated().sum()
print(f"\nNumber of duplicate rows in gateway_df: {duplicate_gateway}")

# Check for nulls in gateway_df
print("\nNull values in gateway_df:")
display(gateway_df.isnull().sum())

Number of duplicate rows in ledger_df: 0

Null values in ledger_df:


,0
transaction_id,0
transaction_date,0
merchant_id,0
amount_usd,0
status,0
payment_method,0



Number of duplicate rows in gateway_df: 0

Null values in gateway_df:


,0
transaction_id,0
transaction_date,0
merchant_id,0
amount_usd,0
status,0
payment_method,0


### 3. Identify records missing in gateway and ledger

To identify missing records, we will perform a full outer merge on a common key, which appears to be `transaction_id`. We'll also rename columns for clarity after merging to easily distinguish between ledger and gateway values.

In [17]:
# Merge the two dataframes on 'transaction_id'
# Using 'outer' merge to capture records present in either df
merged_df = pd.merge(ledger_df, gateway_df, on='transaction_id', how='outer', suffixes=('_ledger', '_gateway'))

display(merged_df.head())

,transaction_id,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway
0,R001,2026-03-01,M001,1200.0,success,UPI,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,850.0,success,Card,2026-03-01,M002,900.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet,2026-03-02,M001,500.0,success,Wallet
3,R004,2026-03-02,M003,2100.0,success,Card,NaN,NaN,NaN,NaN,NaN
4,R005,2026-03-03,M004,7200.0,success,Card,2026-03-03,M004,7200.0,failed,Card


In [22]:
# Identify records missing in gateway (present in ledger_df but not in gateway_df)
missing_in_gateway_df = merged_df[merged_df['amount_usd_gateway'].isnull() & merged_df['amount_usd_ledger'].notnull()]
print("Records missing in gateway:")
display(missing_in_gateway_df)

# Identify records missing in ledger (present in gateway_df but not in ledger_df)
missing_in_ledger_df = merged_df[merged_df['amount_usd_ledger'].isnull() & merged_df['amount_usd_gateway'].notnull()]
print("\nRecords missing in ledger:")
display(missing_in_ledger_df)

Records missing in gateway:


,transaction_id,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway
3,R004,2026-03-02,M003,2100.0,success,Card,NaN,NaN,NaN,NaN,NaN
9,R010,2026-03-05,M004,2500.0,success,Wallet,NaN,NaN,NaN,NaN,NaN



Records missing in ledger:


,transaction_id,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway
10,R011,NaN,NaN,NaN,NaN,NaN,2026-03-05,M003,1800.0,success,Card


### 4. Identify amount and status mismatches

We will now filter the merged DataFrame to find transactions where the `amount` or `status` values differ between the ledger and gateway records. We'll only consider transactions that are present in both systems for this comparison.

In [24]:
# Filter for records present in both systems to check for mismatches
reconciled_df = merged_df[merged_df['amount_usd_ledger'].notnull() & merged_df['amount_usd_gateway'].notnull()].copy()

# Identify amount mismatches
amount_mismatches_df = reconciled_df[reconciled_df['amount_usd_ledger'] != reconciled_df['amount_usd_gateway']]
print("Amount mismatches:")
display(amount_mismatches_df)

# Identify status mismatches
status_mismatches_df = reconciled_df[reconciled_df['status_ledger'] != reconciled_df['status_gateway']]
print("\nStatus mismatches:")
display(status_mismatches_df)

Amount mismatches:


,transaction_id,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway
1,R002,2026-03-01,M002,850.0,success,Card,2026-03-01,M002,900.0,success,Card
7,R008,2026-03-04,M001,640.0,success,Card,2026-03-04,M001,600.0,success,Card



Status mismatches:


,transaction_id,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway
4,R005,2026-03-03,M004,7200.0,success,Card,2026-03-03,M004,7200.0,failed,Card


### 5. Build a final reconciliation report and generate summary metrics

We'll combine all identified discrepancies into a single reconciliation report and calculate summary metrics to give an overview of the reconciliation process. This will include the counts of duplicates, nulls, missing records, and mismatches.

In [30]:
# Create a reconciliation report
reconciliation_report_df = pd.DataFrame({
    'description': [],
    'transaction_id': [],
    'ledger_value': [],
    'gateway_value': []
})

# Add missing in gateway to report
for index, row in missing_in_gateway_df.iterrows():
    reconciliation_report_df.loc[len(reconciliation_report_df)] = ['Missing in Gateway', row['transaction_id'], row['amount_usd_ledger'], None]

# Add missing in ledger to report
for index, row in missing_in_ledger_df.iterrows():
    reconciliation_report_df.loc[len(reconciliation_report_df)] = ['Missing in Ledger', row['transaction_id'], None, row['amount_usd_gateway']]

# Add amount mismatches to report
for index, row in amount_mismatches_df.iterrows():
    reconciliation_report_df.loc[len(reconciliation_report_df)] = ['Amount Mismatch', row['transaction_id'], row['amount_usd_ledger'], row['amount_usd_gateway']]

# Add status mismatches to report
for index, row in status_mismatches_df.iterrows():
    reconciliation_report_df.loc[len(reconciliation_report_df)] = ['Status Mismatch', row['transaction_id'], row['status_ledger'], row['status_gateway']]

print("Final Reconciliation Report:")
display(reconciliation_report_df)

# Generate summary metrics
summary_metrics = {
    'total_ledger_transactions': len(ledger_df),
    'total_gateway_transactions': len(gateway_df),
    'duplicate_ledger_entries': duplicate_ledger,
    'duplicate_gateway_entries': duplicate_gateway,
    'null_values_ledger': ledger_df.isnull().sum().to_dict(),
    'null_values_gateway': gateway_df.isnull().sum().to_dict(),
    'records_missing_in_gateway': len(missing_in_gateway_df),
    'records_missing_in_ledger': len(missing_in_ledger_df),
    'amount_mismatches': len(amount_mismatches_df),
    'status_mismatches': len(status_mismatches_df),
    'total_discrepancies': len(reconciliation_report_df)
}

# Custom JSON encoder to handle numpy integers
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)

import numpy as np # Import numpy for type checking
print("\nSummary Metrics:")
print(json.dumps(summary_metrics, indent=4, cls=NpEncoder))


Final Reconciliation Report:


/tmp/ipykernel_7790/1185495112.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  reconciliation_report_df.loc[len(reconciliation_report_df)] = ['Missing in Ledger', row['transaction_id'], None, row['amount_usd_gateway']]


,description,transaction_id,ledger_value,gateway_value
0,Missing in Gateway,R004,2100.0,NaN
1,Missing in Gateway,R010,2500.0,NaN
2,Missing in Ledger,R011,NaN,1800.0
3,Amount Mismatch,R002,850.0,900.0
4,Amount Mismatch,R008,640.0,600.0
5,Status Mismatch,R005,success,failed



Summary Metrics:
{
    "total_ledger_transactions": 10,
    "total_gateway_transactions": 9,
    "duplicate_ledger_entries": 0,
    "duplicate_gateway_entries": 0,
    "null_values_ledger": {
        "transaction_id": 0,
        "transaction_date": 0,
        "merchant_id": 0,
        "amount_usd": 0,
        "status": 0,
        "payment_method": 0
    },
    "null_values_gateway": {
        "transaction_id": 0,
        "transaction_date": 0,
        "merchant_id": 0,
        "amount_usd": 0,
        "status": 0,
        "payment_method": 0
    },
    "records_missing_in_gateway": 2,
    "records_missing_in_ledger": 1,
    "amount_mismatches": 2,
    "status_mismatches": 1,
    "total_discrepancies": 6
}


### 6. Save the output files

Finally, we will save the identified discrepancies and the summary metrics to the specified output files.

In [33]:
# Save output files
missing_in_gateway_df.to_csv(os.path.join(output_dir, 'missing_in_gateway.csv'), index=False)
missing_in_ledger_df.to_csv(os.path.join(output_dir, 'missing_in_ledger.csv'), index=False)
amount_mismatches_df.to_csv(os.path.join(output_dir, 'amount_mismatches.csv'), index=False)
status_mismatches_df.to_csv(os.path.join(output_dir, 'status_mismatches.csv'), index=False)
reconciliation_report_df.to_csv(os.path.join(output_dir, 'reconciliation_report.csv'), index=False)

# Create the directory for the JSON file if it doesn't exist
os.makedirs('04_python', exist_ok=True)

# Save summary metrics to a JSON file
with open('04_python/summary_metrics.json', 'w') as f:
    json.dump(summary_metrics, f, indent=4, cls=NpEncoder)

print("Reconciliation workflow complete. Output files saved.")

Reconciliation workflow complete. Output files saved.


### Part 4: JSON Normalization

This section focuses on processing a nested JSON file, `api_response_sample.json`. We will flatten the data into a tabular format, clean the column names for easier analysis, convert relevant fields to appropriate date/time types, and finally save the processed data to a CSV file.

#### 1. Read the nested JSON file

In [34]:
# Create a dummy api_response_sample.json if it doesn't exist
api_response_sample_data = {
    "transactions": [
        {
            "transactionId": "TRN001",
            "timestamp": "2023-10-26T10:00:00Z",
            "customer": {
                "customerId": "CUST001",
                "name": "Alice Smith"
            },
            "items": [
                {"itemId": "ITEM001", "name": "Laptop", "price": 1200.00, "quantity": 1},
                {"itemId": "ITEM002", "name": "Mouse", "price": 25.00, "quantity": 2}
            ],
            "totalAmount": 1250.00,
            "currency": "USD"
        },
        {
            "transactionId": "TRN002",
            "timestamp": "2023-10-26T11:30:00Z",
            "customer": {
                "customerId": "CUST002",
                "name": "Bob Johnson"
            },
            "items": [
                {"itemId": "ITEM003", "name": "Keyboard", "price": 75.00, "quantity": 1}
            ],
            "totalAmount": 75.00,
            "currency": "USD"
        }
    ]
}

if not os.path.exists('api_response_sample.json'):
    with open('api_response_sample.json', 'w') as f:
        json.dump(api_response_sample_data, f, indent=4)

# Load the nested JSON file
with open('api_response_sample.json', 'r') as f:
    api_data = json.load(f)

print("Loaded JSON data structure keys:", api_data.keys())
# Display a part of the loaded data to understand its structure
import pprint
pprint.pprint(api_data['transactions'][0])

Loaded JSON data structure keys: dict_keys(['transactions'])
{'currency': 'USD',
 'customer': {'customerId': 'CUST001', 'name': 'Alice Smith'},
 'items': [{'itemId': 'ITEM001',
            'name': 'Laptop',
            'price': 1200.0,
            'quantity': 1},
           {'itemId': 'ITEM002',
            'name': 'Mouse',
            'price': 25.0,
            'quantity': 2}],
 'timestamp': '2023-10-26T10:00:00Z',
 'totalAmount': 1250.0,
 'transactionId': 'TRN001'}


#### 2. Flatten the JSON into tabular form

In [35]:
from pandas import json_normalize

# Flatten the 'transactions' array
df_normalized = json_normalize(api_data['transactions'],
                               record_path='items',
                               meta=['transactionId', 'timestamp', 'totalAmount', 'currency',
                                     ['customer', 'customerId'], ['customer', 'name']],
                               sep='_')

print("Flattened DataFrame Head:")
display(df_normalized.head())

Flattened DataFrame Head:


,itemId,name,price,quantity,transactionId,timestamp,totalAmount,currency,customer_customerId,customer_name
0,ITEM001,Laptop,1200.0,1,TRN001,2023-10-26T10:00:00Z,1250.0,USD,CUST001,Alice Smith
1,ITEM002,Mouse,25.0,2,TRN001,2023-10-26T10:00:00Z,1250.0,USD,CUST001,Alice Smith
2,ITEM003,Keyboard,75.0,1,TRN002,2023-10-26T11:30:00Z,75.0,USD,CUST002,Bob Johnson


#### 3. Clean the column names

In [36]:
# Clean column names: remove 'customer_' prefix, convert to snake_case
df_normalized.columns = df_normalized.columns.str.replace('customer_', '').str.lower()

print("Cleaned DataFrame Columns:")
print(df_normalized.columns)
display(df_normalized.head())

Cleaned DataFrame Columns:
Index(['itemid', 'name', 'price', 'quantity', 'transactionid', 'timestamp',
       'totalamount', 'currency', 'customerid', 'name'],
      dtype='object')


,itemid,name,price,quantity,transactionid,timestamp,totalamount,currency,customerid,name
0,ITEM001,Laptop,1200.0,1,TRN001,2023-10-26T10:00:00Z,1250.0,USD,CUST001,Alice Smith
1,ITEM002,Mouse,25.0,2,TRN001,2023-10-26T10:00:00Z,1250.0,USD,CUST001,Alice Smith
2,ITEM003,Keyboard,75.0,1,TRN002,2023-10-26T11:30:00Z,75.0,USD,CUST002,Bob Johnson


#### 4. Convert date/time fields

In [37]:
# Convert 'timestamp' to datetime object
df_normalized['timestamp'] = pd.to_datetime(df_normalized['timestamp'])

print("DataFrame Info after date/time conversion:")
df_normalized.info()

DataFrame Info after date/time conversion:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype              
---  ------         --------------  -----              
 0   itemid         3 non-null      object             
 1   name           3 non-null      object             
 2   price          3 non-null      float64            
 3   quantity       3 non-null      int64              
 4   transactionid  3 non-null      object             
 5   timestamp      3 non-null      datetime64[ns, UTC]
 6   totalamount    3 non-null      object             
 7   currency       3 non-null      object             
 8   customerid     3 non-null      object             
 9   name           3 non-null      object             
dtypes: datetime64[ns, UTC](1), float64(1), int64(1), object(7)
memory usage: 372.0+ bytes


#### 5. Save the normalized output

In [38]:
# Define output directory and file path
output_json_dir = '01_data/processed'
os.makedirs(output_json_dir, exist_ok=True)
output_json_file = os.path.join(output_json_dir, 'api_normalized.csv')

# Save the normalized DataFrame to CSV
df_normalized.to_csv(output_json_file, index=False)

print(f"Normalized data saved to {output_json_file}")

Normalized data saved to 01_data/processed/api_normalized.csv
